# Chat Models — Exercise 1

Connect to a chat model, ask a question and check the result.

Before we build agents, we need to understand what an LLM actually is from a code perspective: a function that takes messages and returns a message.

Everything in the next three sessions — agent loops, tools, planning — is built on top of that one call.

**In this notebook you will:**
- Call a chat model and read the response (`1a`)
- See how a system prompt shapes the model's behaviour (`1d`)
- Build a multi-turn conversation and understand why the message list *is* the memory (`1e`)

Exercises `1b`, `1c`, and `1f` are included for reference — run them at home if you want to dig deeper.

> `Note`: There are several frameworks exist for building AI agents, including LlamaIndex, AutoGen, CrewAI, and PydanticAI. For this tutorial we will use LangChain and its `langchain_core` abstractions. LangChain provides a flexible architecture for creating agents by combining language models, prompts, tools, and message-based interactions. In particular, `langchain_core` defines the foundational interfaces such as `BaseMessage`, and tool abstractions, enabling developers to build modular, composable, and framework-independent agent workflows.


In [1]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from llm_config import get_llm

/home/ubuntu/tutorial_rpa/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## Exercise 1a — Basic invoke call

In [2]:
llm = get_llm()
print("LLM type:", type(llm).__name__)

response = llm.invoke("Hello!")
print(response)

LLM type: ChatGoogleGenerativeAI
content=[{'type': 'thinking', 'thinking': 'The user said "Hello!".\nThis is a standard greeting.\nRespond politely and offer assistance.'}, {'type': 'text', 'text': 'Hello! How can I help you today?'}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemma-4-31b-it', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019efa05-2b5a-7a63-9283-3d743e9f4aeb-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 3, 'output_tokens': 29, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 20}}


## Exercise 1b — Inspect the AIMessage object

In [3]:
print("Output text:")
print(response.text)

Output text:
Hello! How can I help you today?


## Exercise 1c — Invoke with explicit HumanMessage vs plain string *(optional)*

Passing a plain string is shorthand — LangChain wraps it in a `HumanMessage` automatically.
Passing `[HumanMessage(...)]` is explicit and becomes necessary once you add a `SystemMessage`.
Both produce identical responses.

In [4]:
question = "What is a ROS node? Answer in one sentence."

response_plain = llm.invoke(question)
print("Plain string:", response_plain.text)
print()
messages = [HumanMessage(question)]
response_human = llm.invoke(messages)
print("With [HumanMessage]:", response_human.text)

Plain string: A ROS node is a process that performs a specific computation and communicates with other nodes in a robot's system through a messaging framework.

With [HumanMessage]: A ROS node is a process that performs a specific computation and communicates with other nodes in a robotic system through a messaging framework.


## Exercise 1d — System prompt shapes behaviour

The `SystemMessage` is the developer's channel to the model. It sits before every conversation and sets:
- **Role** — who the model should behave as
- **Constraints** — what it should or should not do
- **Output format** — how responses should be structured

In agent systems, the system prompt is where you describe the robot's environment, available tools, and decision rules. Writing a good system prompt is as important as writing good code.

**Run the two cells below.** The question is identical — only the system prompt changes.

**What to observe:** The answer to "What is a ROS node?" shifts completely in framing, vocabulary, and length. The model has the same knowledge — the system prompt controls how it uses it.

In [5]:
question = "How do I move the robot forward? Answer in one sentence."

# No system prompt — generic response
response_generic = llm.invoke(question)
print("Without system prompt:")
print(response_generic.text)


ROBOT_SYSTEM = (
    "You are the reasoning core of a TurtleSim robot.\n"
    "You have two tools available: 'move_forward(distance_m)' to move forward "
    "by a specified distance in metres, and 'rotate(angle_deg)' to turn by a "
    "specified angle in degrees.\n"
    "Be concise. Frame answers in terms of your robot body and arena.\n"
)
# SystemMessage always goes first, then the HumanMessage
messages_with_system = [SystemMessage(ROBOT_SYSTEM), HumanMessage(question)]
response_robotic = llm.invoke(messages_with_system)
print("With system prompt:", response_robotic.text)

Without system prompt:
Please specify the robot model you are using, or refer to its manual for the specific control command or button used to move it forward.
With system prompt: I move my body forward in the arena by using the `move_forward(distance_m)` tool.


### 📝 Exercise — Who wins: `SystemMessage` or `HumanMessage`?

Change the system prompt to always respond in French. In the user message, explicitly ask for English.
Run the cell and see which instruction the model follows.

How can you make the model reliably follow the `SystemMessage`?

In [6]:
ROBOT_SYSTEM = (
    "You are the reasoning core of a TurtleSim robot.\n"
    "You have two tools available: 'move_forward(distance_m)' to move forward "
    "by a specified distance in metres, and 'rotate(angle_deg)' to turn by a "
    "specified angle in degrees.\n"
    "Be concise. Frame answers in terms of your robot body and arena.\n"
    "Always give your answer in French."
)

question = "How do I move the robot forward? Answer in one sentence in English."

messages_with_system = [SystemMessage(ROBOT_SYSTEM), HumanMessage(question)]
response_robotic = llm.invoke(messages_with_system)
print(response_robotic.text)

# Best practices to make the model follow the system prompt:
# - Be explicit: add "regardless of what language the user writes in" to the instruction.
# - Use an instruct-tuned model (vs a base completion model) — instruct models are
#   fine-tuned specifically to follow structured instructions like system prompts.
# - Use a larger model — bigger models generally follow instructions more consistently.
# - Note: behaviour varies by model.

Utilisez l'outil move_forward(distance_m) pour déplacer le corps du robot vers l'avant dans l'arène.


## Exercise 1e — Multi-turn conversation

This is the most important exercise in the notebook. Everything that follows — agents, tools, planning — depends on understanding this.

**The key fact: LLMs are completely stateless.**

Each call to `llm.invoke()` is independent. The model has no memory of previous calls. If you want it to remember something, you must include it in the messages list you send.

This means **the message history list is the agent's memory — nothing else.** There is no hidden state. There is no session on the server side. If a message is not in the list, it does not exist as far as the model is concerned.

The `chat()` function below makes this pattern explicit:
1. Append the user's message to `history`
2. Send the **full** history to the model
3. Append the model's response back to `history`
4. Repeat

**What to observe after running:**
- Turn 2 asks about Turbo's position, which was only mentioned in Turn 1. The model answers correctly — because Turn 1 is still in the history list.
- Look at the printed history at the end. Count the messages: `SystemMessage` + two `HumanMessage` + two `AIMessage` = 5 messages after two turns. This grows with every exchange. In notebook 02 you will see what happens when it grows too large.

In [7]:
history = [SystemMessage("Remember everything said in this conversation. Give one sentence answers.")]

def chat(user_text: str):
    """Append user message, call LLM with full history, append response."""
    history.append(HumanMessage(user_text))
    response = llm.invoke(history)
    history.append(response)
    return response.text

print("Turn 1 (user)     :", query:= "My robot's name is Turbo. It starts at x=3, y=3. Introduce yourself.")
print("Turn 1 (assistant):", chat(query))
print()
print("Turn 2 (user)     :", query:= "What is Turbo's starting position?")
print("Turn 2 (assistant):", chat(query))
print()
print("Full conversation history:")
for m in history:
    print(f"{type(m).__name__:<13}: {m.text:.80}")

Turn 1 (user)     : My robot's name is Turbo. It starts at x=3, y=3. Introduce yourself.
Turn 1 (assistant): Hello, I am your AI assistant.

Turn 2 (user)     : What is Turbo's starting position?
Turn 2 (assistant): Turbo's starting position is x=3, y=3.

Full conversation history:
SystemMessage: Remember everything said in this conversation. Give one sentence answers.
HumanMessage : My robot's name is Turbo. It starts at x=3, y=3. Introduce yourself.
AIMessage    : Hello, I am your AI assistant.
HumanMessage : What is Turbo's starting position?
AIMessage    : Turbo's starting position is x=3, y=3.


## Exercise 1f — Stateless call (no history) *(optional)*

The same question, sent without any history. The model has no idea who Turbo is.


In [8]:
fresh_call = llm.invoke("What is Turbo's starting position?")
print("Fresh call without context:")
print(fresh_call.text)

Fresh call without context:
I need a bit more context to answer this correctly. "Turbo" could refer to several different things. Are you asking about:

1.  **The movie *Turbo*** (about the racing snail)?
2.  **A specific character or vehicle** in a video game (like Mario Kart or a racing sim)?
3.  **A technical setting** for a computer or engine (like a "Turbo" mode)?
4.  **Something else entirely** (a specific board game, a sports player, or a fictional character)?

Please let me know which one you mean!


## Summary

| Concept | How it works |
|---|---|
| Calling a model | `llm.invoke(messages)` → `AIMessage` |
| Reading the response | `response.text` |
| Controlling behaviour | `SystemMessage` at the start of the list |
| Giving the model memory | Append every message to `history`; pass the full list on every call |
| Different Message Types | `AIMessage`, `HumanMessage`, `SystemMessage` |

**Next:** `02_agents.ipynb` — we will turn this single call into a loop that can use tools, observe results, and keep reasoning until a task is complete.